<img src="images/header.png" width=180, align="center"/>

Master's degree in Intelligent Systems

Subject: 11754 - Deep Learning

Year: 2025-2026

Professor: Miguel Ángel Calafat Torrens

# LESSON 3A — GANs: Fundamentals and Classical GAN

Generative Adversarial Networks (GANs) were a groundbreaking approach in deep learning that revolutionized the field of artificial intelligence. GANs consist of two neural networks — a generator and a discriminator — engaged in a competitive game. The generator aims to create realistic data samples, while the discriminator distinguishes between real and fake samples. Through adversarial training, GANs generate increasingly realistic data.

This notebook covers GAN theory, a warm-up on MNIST to build intuition, and a full classical GAN implementation on the CelebA face dataset. We'll also explore common failure modes: training instability and mode collapse.

In [ ]:
# Environment detection: Colab vs local
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running on {"Google Colab" if IN_COLAB else "local environment"}')

In [ ]:
# Setup: Drive mount (Colab) or local path
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/gdrive')
    %cd '/content/gdrive/MyDrive/LABS2026/LAB03'

import pathlib
import sys
import os

PROJECT_DIR = str(pathlib.Path().resolve())
sys.path.append(PROJECT_DIR)

import helper_L3 as hp

In [ ]:
# Imports
import PIL
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)

# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("mps")

print(f"Using device: {DEVICE}")

# 1. Introduction to GANs

## What are GANs?

A Generative Adversarial Network consists of two neural networks trained simultaneously in a competitive setting:

- **Generator (G)**: Takes random noise as input and produces synthetic data (e.g., images). Its goal is to create samples that are indistinguishable from real data.
- **Discriminator (D)**: Takes an input (either real or generated) and outputs a probability that the input is real. Its goal is to correctly distinguish real data from fakes.

**Analogy**: Think of a counterfeiter (generator) trying to produce fake currency and a detective (discriminator) trying to spot the fakes. Over time, the counterfeiter gets better at making realistic fakes, while the detective gets better at detecting them. The competition drives both to improve.

<img src="images/gan_architecture.png" width=600, align="center"/>

## The Minimax Objective

The GAN training is formalized as a minimax game:

$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

Breaking this down:
- $\mathbb{E}_{x \sim p_{data}}[\log D(x)]$: The discriminator wants to assign high probability to real data → maximize this term.
- $\mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$: The discriminator wants to assign low probability to fake data → maximize this term. The generator wants the opposite → minimize it.

In practice, we train D to maximize $V$ and G to minimize it, alternating between the two.

**Note:** Rather than minimizing $\log(1 - D(G(z)))$ for the generator (which saturates when $D$ is confident), we maximize $\log(D(G(z)))$ instead. This provides stronger gradients early in training when the generator is still poor — the so-called "non-saturating" variant from the original GAN paper.

<img src="images/minimax_objective.png" width=600, align="center"/>

## The Latent Space

The input to the generator is a vector $z$ sampled from a simple distribution (typically a standard normal distribution, $z \sim \mathcal{N}(0, I)$). This vector lives in what we call the **latent space**.

Key concepts:
- The generator learns a mapping $G: \mathcal{Z} \rightarrow \mathcal{X}$ from the simple latent distribution to the complex data distribution.
- **Interpolation**: If the generator has learned a smooth mapping, interpolating between two latent vectors $z_1$ and $z_2$ should produce a smooth transition between the corresponding generated images. This is a sign of a well-structured latent space.
- **Dimensionality**: The latent dimension (`z_dim`) controls the "capacity" of the representation. Too small → not enough variation. Too large → harder to train.

# 2. GAN Warm-Up with MNIST

Let's start with MNIST (familiar from LAB02) to see the GAN game in action before tackling harder data. We'll build a simple fully-connected GAN (FC-GAN) — no convolutions, just linear layers. This trains fast and gives a clear demonstration of the adversarial game.

In [ ]:
# Load MNIST dataset
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Scale to [-1, 1]
])

mnist_dataset = datasets.MNIST(root='data', train=True, download=True,
                               transform=mnist_transform)
mnist_loader = DataLoader(mnist_dataset, batch_size=128, shuffle=True)

In [ ]:
# FC-GAN Generator: maps z_dim -> 784 (28x28) image
class Generator_FC(nn.Module):
    def __init__(self, z_dim=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 784),
            nn.Tanh()
        )

    def forward(self, x):
        out = self.net(x)
        return out.view(-1, 1, 28, 28)

In [ ]:
# FC-GAN Discriminator: maps 784 -> 1 (real/fake score)
class Discriminator_FC(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1)
            # No final activation — we use BCEWithLogitsLoss
        )

    def forward(self, x):
        return self.net(x.view(-1, 784))

In [ ]:
# Instantiate models, optimizers, and loss
torch.manual_seed(SEED)
G_fc = Generator_FC(z_dim=100).to(DEVICE)
D_fc = Discriminator_FC().to(DEVICE)
g_opt_fc = torch.optim.Adam(G_fc.parameters(), lr=0.0002, betas=(0.5, 0.999))
d_opt_fc = torch.optim.Adam(D_fc.parameters(), lr=0.0002, betas=(0.5, 0.999))
criterion_fc = nn.BCEWithLogitsLoss()

In [ ]:
# Train the FC-GAN on MNIST (~5 minutes on GPU)
n_epochs_fc = 20
z_dim_fc = 100
g_losses_fc, d_losses_fc = [], []

for epoch in range(1, n_epochs_fc + 1):
    g_loss_epoch, d_loss_epoch = 0.0, 0.0
    n_batches = 0

    for real_imgs, _ in tqdm(mnist_loader, desc=f"Epoch {epoch}"):
        n_batches += 1
        bs = real_imgs.size(0)
        real_imgs = real_imgs.to(DEVICE)

        # --- Train Discriminator ---
        d_opt_fc.zero_grad()
        # Real
        d_real = D_fc(real_imgs)
        d_real_loss = criterion_fc(d_real.squeeze(), torch.ones(bs, device=DEVICE))
        # Fake
        noise = torch.randn(bs, z_dim_fc, device=DEVICE)
        fake_imgs = G_fc(noise)
        d_fake = D_fc(fake_imgs.detach())
        d_fake_loss = criterion_fc(d_fake.squeeze(), torch.zeros(bs, device=DEVICE))
        d_loss = d_real_loss + d_fake_loss
        d_loss.backward()
        d_opt_fc.step()

        # --- Train Generator ---
        g_opt_fc.zero_grad()
        noise = torch.randn(bs, z_dim_fc, device=DEVICE)
        fake_imgs = G_fc(noise)
        d_fake = D_fc(fake_imgs)
        g_loss = criterion_fc(d_fake.squeeze(), torch.ones(bs, device=DEVICE))
        g_loss.backward()
        g_opt_fc.step()

        g_loss_epoch += g_loss.item()
        d_loss_epoch += d_loss.item()

    g_losses_fc.append(g_loss_epoch / n_batches)
    d_losses_fc.append(d_loss_epoch / n_batches)
    print(f"  G loss: {g_losses_fc[-1]:.4f}, D loss: {d_losses_fc[-1]:.4f}")

In [ ]:
# Visualize generated digits
G_fc.eval()
with torch.no_grad():
    noise = torch.randn(25, z_dim_fc, device=DEVICE)
    generated = G_fc(noise).cpu()
    # Scale from [-1,1] to [0,1] for display
    generated = (generated + 1) * 0.5

fig, axes = plt.subplots(5, 5, figsize=(7, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle("FC-GAN Generated Digits (MNIST)")
plt.tight_layout()
plt.show()

# Save healthy FC-GAN for later mode collapse comparison
healthy_fc_generated = generated.clone()

Even with this simple fully-connected architecture, the generator learned to produce digit-like images. The quality is limited (blurry, sometimes malformed) because we used only linear layers — no spatial structure. In the next section, we'll use convolutional architectures on a much harder dataset.

# 3. Classical GAN on CelebA

## The CelebA Dataset

In this section we will use the CelebA dataset. You can find it in a lot of places all over the internet, but one good place to start is its own [website](http://mmlab.ie.cuhk.edu.hk/projects/CelebA.html) where you scroll down to the "**Download**" section, select the link "**In the wild images**" go into the folder "**Img**" and download the file "**image_align_celeba.zip**". Maybe it's even easier download it from this [link of Kaggle](https://www.kaggle.com/jessicali9530/celeba-dataset) if you have an account (the account is free).

Anyway, you just remember that **the CelebA dataset is available for non-commercial research purposes only**.

The dataset is 1.34GB in size because it has more than 200,000 images. In this practice, since what is done is a proof of concept, you do not need to upload all the images. A good option would be to take a selection of, for example, 10,000 photos and compress them into a file that will be the one you upload to your Drive account.

In my case I've taken the first 10,000 images of CelebA and compressed them into a file called `img_align_celeba_small_10000.zip` that I've uploaded to my GDrive.

In [ ]:
# Ensure you write down the correct path to your zip file with the dataset

if IN_COLAB:
    # CelebA full
    # dataset_zip_fullpath = '/content/gdrive/MyDrive/datasets/img_align_celeba.zip'

    # CelebA small
    dataset_zip_fullpath = '/content/gdrive/MyDrive/datasets/img_align_celeba_small_10000.zip'
else:
    # Local path — adjust to your setup
    dataset_zip_fullpath = os.path.join(PROJECT_DIR, '..', 'datasets', 'img_align_celeba_small_10000.zip')


**Before running the next cell, make sure you have left the zip file in the location you want and assign the variable accordingly.**

In [ ]:
# This cell will take several minutes the first time you run it.
# You can execute this cell every time you run the code. It won't unzip
# anything if the dataset is already unzipped.
DATA_DIR = hp.extract_dataset(dataset_zip_fullpath, remove_zip=IN_COLAB)


## The Dataset Class

The dataset is made up of images of faces of famous people. Taking into account that we do not want to do very expensive computational training, we are not interested in using the images with their original resolution (218 x 178), but it is better for us to convert them to a lower resolution, which in this case will be 64 x 64.

Note that a custom dataset class is defined in the next cell. It uses the `__getitem__()` method to implement the corresponding transformations. In addition to the image, a label is returned, which in this case is not necessary. The label is the name of the file.

In [ ]:
# CustomDataset is defined here for teaching purposes — see the implementation.
# It is also available as hp.CustomDataset in helper_L3.py for reuse.
class CustomDataset(Dataset):
    def __init__(self, img_folder, lim=-1, transforms=None):
        # lim is the max number of images to load (-1 to load all images).
        # The default is intended for proof-of-concept runs so that we don't
        # load the entire dataset. Use lim=-1 to load all images.
        self.img_folder = img_folder
        self.lim = lim

        # Initialize empty lists for items and labels
        self.items = []
        self.labels = []

        # Walk through all files in img_folder and its subfolders
        # Sort directories and files for deterministic ordering
        for root, dirs, files in os.walk(img_folder):
            dirs.sort()
            for file in sorted(files):
                # Check if the file is an image
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    full_path = os.path.join(root, file)
                    self.items.append(full_path)
                    self.labels.append(file)

                    # Stop if the limit is reached
                    if lim > 0 and len(self.items) >= lim:
                        break

            if lim > 0 and len(self.items) >= lim:
                break

        # Define the transformation pipeline
        self.transforms = transforms

    def __getitem__(self, idx):
        # Load the image
        data = PIL.Image.open(self.items[idx]).convert('RGB')

        # Apply the transformations if provided
        if self.transforms:
            data = self.transforms(data)

        # Return the processed data and its corresponding label
        return data, self.labels[idx]

    def __len__(self):
        return len(self.items)

## Image Transformations

The generator typically uses a `tanh` activation function, whose output gives a value between -1 and 1. For this reason it is recommended that the input tensors are already scaled in this range.

In [ ]:
# ToScaledTensor extends ToTensor to scale to [-1, 1].
# Also available as hp.ToScaledTensor in helper_L3.py for reuse.
class ToScaledTensor(transforms.ToTensor):
    def __init__(self, low=-1, high=1):
        super().__init__()
        self.low = low
        self.high = high

    def __call__(self, img):
        # Convert image to tensor (same as ToTensor)
        tensor = super().__call__(img)
        # Scale the tensor to the desired range
        tensor = tensor * (self.high - self.low) + self.low
        return tensor

And now we can define the transformations in the usual way. The first step is to crop the original image (218 x 178) to create a centered square image (178 x 178), which is the largest size possible. This image does not crop faces in the vast majority of cases.
Then we will resize the image to the required dimension (64 x 64).

In [ ]:
# Size of the images (height and width)
img_size = 64

# Transformations
transform = transforms.Compose([
    # Center crop the images so that they become square
    transforms.CenterCrop((178, 178)),
    # Resize the image to the specified size (h, w)
    transforms.Resize((img_size, img_size)),
    # Convert the image to a PyTorch tensor and scale to [-1, 1]
    ToScaledTensor(),
])

In [ ]:
# Max number of images to use (-1 if all)
# Maybe you want to start working with fewer images to see how it works
limit = -1

# Define the custom dataset object
dataset = CustomDataset(DATA_DIR, transforms=transform, lim=limit)

In [ ]:
# Define the DataLoader
batch_size = 128
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# Let's see images of the first batch
hp.show(next(iter(dataloader))[0])

## The Networks

Now the time has come to define the two competing models. On the one hand we have the generator, which will have a decoder configuration, and on the other hand we will have the discriminator, which will have an encoder configuration.

They can be considered symmetrical networks because one attempts to generate a credible image from a latent space, while the other analyzes an image to determine its authenticity, mapping it to a single score.

<img src="images/network_architecture.png" width=900, align="center"/>

**Note:** Following DCGAN conventions, we do **not** apply batch normalization to the first convolutional layer of the discriminator. This is standard practice — the first layer benefits from seeing raw pixel statistics. Notice that in our implementation we are not using the dimensions above, instead we are using the ones seen below.

<img src="images/architecture_dimensions.png" width=700, align="center"/>

In [ ]:
# Discriminator with an encoder structure of 4 downscale steps.
# Note: No normalization on the first convolutional layer (DCGAN convention).
class Discriminator(nn.Module):
    def __init__(self, d_dim=64, img_size=64):
        # "d_dim" is the output dimension of the first convolutional layer;
        # that is, the number of filters/kernels you have in the first layer
        # (3 inputs, channels RGB, and d_dim outputs)
        super().__init__()

        # Configuration parameters
        kernel_size = 4
        n = 4  # Number of conv layers. Only used for definition of fc_in
        fc_in = int(d_dim * 2**(n-1) * (img_size/(2**n))**2)
        pad = 1
        stride = 2
        bias = False

        # Helper function for convolutions
        def conv(in_channels, out_channels):
            return nn.Conv2d(in_channels=in_channels,
                             out_channels=out_channels,
                             kernel_size=kernel_size,
                             stride=stride,
                             padding=pad,
                             bias=bias)

        # Dense or fully connected layer
        self.fc = nn.Linear(fc_in, 1)

        # Convolutional layers (doubling the depth with each step)
        self.conv1 = conv(in_channels=3, out_channels=d_dim)
        self.conv2 = conv(in_channels=d_dim, out_channels=2*d_dim)
        self.conv3 = conv(in_channels=2*d_dim, out_channels=4*d_dim)
        self.conv4 = conv(in_channels=4*d_dim, out_channels=8*d_dim)

        # Batchnorm layers. Not applied to the first convolutional layer
        self.bnorm2 = nn.BatchNorm2d(2 * d_dim)
        self.bnorm3 = nn.BatchNorm2d(4 * d_dim)
        self.bnorm4 = nn.BatchNorm2d(8 * d_dim)

        # Define a LeakyReLU activation function layer
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, x):
        # Input shape: batch x 3 x img_size x img_size  -->  b x 3 x 64 x 64
        batch_size = x.size(0)

        out = self.leaky_relu(self.conv1(x))                 # b x 64 x 32 x 32
        out = self.leaky_relu(self.bnorm2(self.conv2(out)))  # b x 128 x 16 x 16
        out = self.leaky_relu(self.bnorm3(self.conv3(out)))  # b x 256 x 8 x 8
        out = self.leaky_relu(self.bnorm4(self.conv4(out)))  # b x 512 x 4 x 4

        # Flatten
        out = out.contiguous().view(batch_size, -1)  # b x 8192

        # Final output layer without activation function
        scores = self.fc(out)  # b x 1
        return scores

The generator has the reversed configuration. We start with a latent code of 100 values, which is converted via a fully connected layer into a tensor of 8192 positions. Afterwards, it is viewed as a (512, 4, 4) tensor and then upsampled back to image dimensions using 4 steps of transposed convolutional layers.

<img src="images/generator_architecture.png" width=700, align="center"/>

In [ ]:
class Generator(nn.Module):
    # 'd_dim' is the number of input channels of the last transposed convolutional layer;
    # 'z_dim' is the size of the input to the network (latent space dimension).

    def __init__(self, z_dim=100, d_dim=64, img_size=64):
        super().__init__()

        # Configuration parameters
        kernel_size = 4
        self.d_dim = d_dim
        n = 4

        # Calculate dimensions of fully connected layer
        fc_out = int(d_dim * 2**(n-1) * (img_size/(2**n))**2)
        self.i_s_fco = int(img_size / 2**n)

        pad = 1
        stride = 2
        bias = False

        def tconv(in_channels, out_channels):
            return nn.ConvTranspose2d(in_channels=in_channels,
                                      out_channels=out_channels,
                                      kernel_size=kernel_size,
                                      stride=stride,
                                      padding=pad,
                                      bias=bias)

        # Linear layer
        self.fc = nn.Linear(z_dim, fc_out)

        # Transpose convolutional layers
        self.tconv1 = tconv(in_channels=d_dim*8, out_channels=d_dim*4)
        self.tconv2 = tconv(in_channels=d_dim*4, out_channels=d_dim*2)
        self.tconv3 = tconv(in_channels=d_dim*2, out_channels=d_dim)
        self.tconv4 = tconv(in_channels=d_dim, out_channels=3)

        # Batchnorm layers
        self.bnorm1 = nn.BatchNorm2d(d_dim * 4)
        self.bnorm2 = nn.BatchNorm2d(d_dim * 2)
        self.bnorm3 = nn.BatchNorm2d(d_dim)

        # Activation function layers
        self.relu = nn.ReLU(True)
        self.tanh = nn.Tanh()

    def forward(self, x):
        b = x.size(0)

        out = self.fc(x)                                # b x 8192
        out = out.contiguous().view(b, -1,
                                    self.i_s_fco,
                                    self.i_s_fco)       # b x 512 x 4 x 4

        out = self.relu(self.bnorm1(self.tconv1(out)))  # b x 256 x 8 x 8
        out = self.relu(self.bnorm2(self.tconv2(out)))  # b x 128 x 16 x 16
        out = self.relu(self.bnorm3(self.tconv3(out)))  # b x 64 x 32 x 32
        out = self.tanh(self.tconv4(out))               # b x 3 x 64 x 64

        return out

In [ ]:
# Instantiate the models with default parameters
D = Discriminator()
G = Generator()

## Optimizers

In this configuration, both networks share the same optimizer hyperparameters: learning rate and beta values. This is not always the case — different hyperparameters for D and G are sometimes used.

In [ ]:
# Discriminator optimizer
lr_d = 0.0004
beta1 = 0.6
beta2 = 0.999
d_optimizer = torch.optim.Adam(D.parameters(), lr_d, [beta1, beta2])

In [ ]:
# Generator optimizer
lr_g = 0.0004
beta1 = 0.6
beta2 = 0.999
g_optimizer = torch.optim.Adam(G.parameters(), lr_g, [beta1, beta2])

## Loss Function

The GAN loss function is based on Binary Cross Entropy. Because the discriminator's output lacks an activation function, we use `BCEWithLogitsLoss`, which applies the sigmoid internally for numerical stability.

This connects directly to the minimax objective from Section 1: when we train the discriminator with real images, we want $D(x) \approx 1$; with fake images, $D(G(z)) \approx 0$. The generator tries to make $D(G(z)) \approx 1$.

In [ ]:
# Define the loss function.
# Also available as hp.gan_loss_fcn in helper_L3.py for reuse.
def gan_loss_fcn(discr_output, **kwargs):
    """
    Calculates the loss based on specified label type (real or fake).

    Args:
        discr_output: Tensor of discriminator logits.
        **kwargs: label_type (str): "real" or "fake"; smooth (bool).

    Returns:
        Tensor containing the calculated loss.
    """
    batch_size = discr_output.size(0)

    label_type = kwargs.get("label_type", "real").lower()
    smooth = kwargs.get("smooth", False)

    if label_type not in ("real", "fake"):
        raise ValueError(
            f"label_type must be 'real' or 'fake', got '{label_type}'")

    if label_type == "real":
        labels = torch.ones(batch_size) * (0.9 if smooth else 1.0)
    else:
        labels = torch.zeros(batch_size)

    labels = labels.to(DEVICE)

    # Note: the helper version uses a module-level criterion to avoid
    # re-instantiation on each call. This inline version is simpler for teaching.
    criterion = nn.BCEWithLogitsLoss()
    loss = criterion(discr_output.squeeze(), labels)

    return loss

## Checkpoints

As usual, we're going to perform checkpoints. The function below will be called within the training function. It will assess whether a checkpoint is required, and if so, it will save using the helper function from `helper_L3.py`.

In [ ]:
# Checkpointer function — saves checkpoints based on loss improvement or interval.
# Also available as hp.checkpointer in helper_L3.py for reuse.
def checkpointer(epoch, epoch_gener_loss, best_gen_loss, config,
                 save_step, starting_from=20):
    """
    Conditionally save a checkpoint. Returns updated best_gen_loss.
    """
    save_path = config.get('project_dir', '.')
    prefix = config.get('checkpoint_prefix', '')
    if prefix:
        prefix = f"{prefix}_"


    if epoch_gener_loss < best_gen_loss and epoch > starting_from:
        best_gen_loss = epoch_gener_loss
        print(f"New best generator loss {best_gen_loss} at epoch"
              f" {epoch}. Saving checkpoint.")
        hp.save_checkpoint(f"{prefix}best_{epoch}", epoch, config, save_path)

    elif epoch % save_step == 0:
        print(f"Epoch {epoch}. "
              "Not best losses, but saving checkpoint anyway.")
        hp.save_checkpoint(f"{prefix}epoch_{epoch}", epoch, config, save_path)

    return best_gen_loss

## Training

At this point, we reach the training phase, which involves training the two models in competition. Within the loop through batches, both the discriminator and the generator are trained. Moreover, within this same sweep, a new loop is introduced that could train the discriminator more times than the generator.

<img src="images/training_loop.png" width=900, align="center"/>

<img src="images/training_schematic.png" width=500, align="center"/>

In [ ]:
# Training function — also available as hp.train in helper_L3.py for reuse.
def train(config, verbose=True):
    """Training loop for GAN / WGAN-GP."""
    best_gen_loss = float('inf')
    gener_losses_epoch_list = []
    discr_losses_epoch_list = []

    n_epochs = config.get('n_epochs', 100)
    crit_cycles = config.get('crit_cycles', 1)
    z_dim = config.get('z_dim', 100)
    show_step = config.get('show_step', 25)
    save_step = config.get('save_step', 5)
    last_epoch = config.get('epoch', 0)
    save_starting = config.get('save_starting', 20)
    gp_fcn = config.get('penalty_fcn', lambda *x: 0)

    dataloader = config['dataloader']
    gener = config['generator'].to(DEVICE)
    discr = config['discriminator'].to(DEVICE)
    gener_opt = config['g_optimizer']
    discr_opt = config['d_optimizer']
    loss_fcn = config['loss_fcn']

    for epoch in range(last_epoch + 1, n_epochs + last_epoch + 1):
        epoch_gener_loss = 0.0
        epoch_discr_loss = 0.0
        num_batches = 0

        for real_imgs, _ in tqdm(dataloader):
            num_batches += 1
            current_bs = len(real_imgs)
            real_imgs = real_imgs.to(DEVICE)

            # Train discriminator/critic
            discr_loss_for_cycles = 0
            for _ in range(crit_cycles):
                discr_opt.zero_grad()
                noise = torch.randn(current_bs, z_dim, device=DEVICE)
                fake_imgs = gener(noise)
                discr_fake_pred = discr(fake_imgs.detach())
                discr_fake_loss = loss_fcn(discr_fake_pred, label_type='fake')
                discr_real_pred = discr(real_imgs)
                discr_real_loss = loss_fcn(discr_real_pred, label_type='real')
                penalty = gp_fcn(real_imgs, fake_imgs.detach(), discr)
                discr_loss = discr_fake_loss + discr_real_loss + penalty
                discr_loss_for_cycles += discr_loss.item() / crit_cycles
                discr_loss.backward()
                discr_opt.step()

            epoch_discr_loss += discr_loss_for_cycles

            # Train generator
            gener_opt.zero_grad()
            noise = torch.randn(current_bs, z_dim, device=DEVICE)
            fake_imgs = gener(noise)
            discr_fake_pred = discr(fake_imgs)
            gener_loss = loss_fcn(discr_fake_pred, label_type='real')
            epoch_gener_loss += gener_loss.item()
            gener_loss.backward()
            gener_opt.step()

        epoch_gener_loss /= num_batches
        epoch_discr_loss /= num_batches
        gener_losses_epoch_list.append(epoch_gener_loss)
        discr_losses_epoch_list.append(epoch_discr_loss)

        if verbose:
            print({'Epoch': epoch,
                   'D/C loss': epoch_discr_loss,
                   'Gen loss': epoch_gener_loss})

        # Checkpoint — capture updated best_gen_loss
        best_gen_loss = checkpointer(
            epoch=epoch,
            epoch_gener_loss=epoch_gener_loss,
            best_gen_loss=best_gen_loss,
            config=config,
            save_step=save_step,
            starting_from=save_starting)

        if epoch % show_step == 0:
            hp.visual_epoch(fake_imgs, real_imgs,
                            gener_losses_epoch_list,
                            discr_losses_epoch_list)

    return gener, discr, [gener_losses_epoch_list, discr_losses_epoch_list]

In [ ]:
# Configure and train the classical GAN — 40 epochs
# We save checkpoints at epoch 40 for comparison with WGAN-GP in LESSON_3B
n_epochs = 40

config = {'n_epochs': n_epochs,
          'dataloader': dataloader,
          'discriminator': D,
          'generator': G,
          'd_optimizer': d_optimizer,
          'g_optimizer': g_optimizer,
          'project_dir': PROJECT_DIR,
          'loss_fcn': gan_loss_fcn,
          'show_step': 10,
          'save_step': 40,
          'save_starting': 1000,
          'checkpoint_prefix': 'classical',
          }

In [ ]:
_ = train(config)

# 4. What Goes Wrong with Classical GANs

GANs are notoriously hard to train. Small changes in hyperparameters can cause catastrophic failure. Let's see what happens when we make poor choices.

## Training Instability Demo

We demonstrate instability on MNIST using the FC-GAN from Section 2 — this allows us to train many more epochs quickly and see the chaotic loss curves clearly. Let's see what happens with bad hyperparameters — a learning rate that's too high and standard beta1=0.9 (instead of the GAN-recommended ~0.5-0.6).

In [ ]:
# Instability demo: FC-GAN on MNIST with BAD hyperparameters
torch.manual_seed(SEED)
z_dim_bad = 100
G_bad = Generator_FC(z_dim=z_dim_bad).to(DEVICE)
D_bad = Discriminator_FC().to(DEVICE)

# BAD hyperparameters: lr too high, beta1 too close to 1
g_opt_bad = torch.optim.Adam(G_bad.parameters(), lr=0.01, betas=(0.9, 0.999))
d_opt_bad = torch.optim.Adam(D_bad.parameters(), lr=0.01, betas=(0.9, 0.999))

n_epochs_bad = 50
g_losses_bad, d_losses_bad = [], []

G_bad.train()
D_bad.train()

for epoch in range(1, n_epochs_bad + 1):
    g_loss_epoch, d_loss_epoch = 0.0, 0.0
    n_batches = 0

    for real_imgs, _ in tqdm(mnist_loader, desc=f"Bad epoch {epoch}"):
        n_batches += 1
        bs = real_imgs.size(0)
        real_imgs = real_imgs.to(DEVICE)

        # Train D
        d_opt_bad.zero_grad()
        d_real = D_bad(real_imgs)
        d_real_loss = criterion_fc(d_real.squeeze(), torch.ones(bs, device=DEVICE))
        noise = torch.randn(bs, z_dim_bad, device=DEVICE)
        fake_imgs = G_bad(noise)
        d_fake = D_bad(fake_imgs.detach())
        d_fake_loss = criterion_fc(d_fake.squeeze(), torch.zeros(bs, device=DEVICE))
        d_loss = d_real_loss + d_fake_loss
        d_loss.backward()
        d_opt_bad.step()

        # Train G
        g_opt_bad.zero_grad()
        noise = torch.randn(bs, z_dim_bad, device=DEVICE)
        fake_imgs = G_bad(noise)
        d_fake = D_bad(fake_imgs)
        g_loss = criterion_fc(d_fake.squeeze(), torch.ones(bs, device=DEVICE))
        g_loss.backward()
        g_opt_bad.step()

        g_loss_epoch += g_loss.item()
        d_loss_epoch += d_loss.item()

    g_losses_bad.append(g_loss_epoch / n_batches)
    d_losses_bad.append(d_loss_epoch / n_batches)

# Plot loss curves + generated samples
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(g_losses_bad, label='Generator')
ax1.plot(d_losses_bad, label='Discriminator')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Instability: Chaotic Loss Curves')
ax1.legend()

n_display = 25
G_bad.eval()
with torch.no_grad():
    noise = torch.randn(n_display, z_dim_bad, device=DEVICE)
    bad_imgs = G_bad(noise).cpu()
    bad_imgs = (bad_imgs + 1) * 0.5
grid_bad = make_grid(bad_imgs, nrow=5).permute(1, 2, 0)
ax2.imshow(grid_bad.clip(0, 1))
ax2.set_title('Generated Images (Bad Hyperparameters)')
ax2.axis('off')

plt.tight_layout()
plt.show()

Notice the chaotic loss curves — with 50 epochs, the instability pattern is clearly visible. Unlike our healthy FC-GAN training in Section 2, the losses never stabilize. The generated images are noise-like garbage despite training on the same MNIST dataset that produced recognizable digits earlier. This is caused by two bad hyperparameter choices: **(1)** the learning rate is too high (0.01 vs the recommended ~0.0002), causing the models to overshoot optimal parameters on every update; and **(2)** beta1=0.9 retains too much gradient momentum, amplifying the instability instead of dampening it (the GAN-recommended ~0.5-0.6 provides more stable updates).

# 5. Mode Collapse

Sometimes the generator finds a "shortcut" — it produces only a few types of outputs that
fool the discriminator. The generator "collapses" to a small number of modes of the data
distribution.

Mode collapse is easier to observe on MNIST, where the modes correspond to digit classes
(0-9). We can force mode collapse by using a very small latent space (`z_dim=2`). With only
2 dimensions, the generator cannot represent all 10 digit classes and is forced to collapse
to a subset.

In [ ]:
# Mode collapse demo: FC-GAN with z_dim=2 (too small to represent 10 digit classes)
torch.manual_seed(SEED)
G_collapse = Generator_FC(z_dim=2).to(DEVICE)
D_collapse = Discriminator_FC().to(DEVICE)
g_opt_collapse = torch.optim.Adam(G_collapse.parameters(), lr=0.0002, betas=(0.5, 0.999))
d_opt_collapse = torch.optim.Adam(D_collapse.parameters(), lr=0.0002, betas=(0.5, 0.999))

n_epochs_collapse = 20
for epoch in range(1, n_epochs_collapse + 1):
    for real_imgs, _ in tqdm(mnist_loader, desc=f"Collapse epoch {epoch}"):
        bs = real_imgs.size(0)
        real_imgs = real_imgs.to(DEVICE)

        # Train D
        d_opt_collapse.zero_grad()
        d_real = D_collapse(real_imgs)
        d_real_loss = criterion_fc(d_real.squeeze(), torch.ones(bs, device=DEVICE))
        noise = torch.randn(bs, 2, device=DEVICE)
        fake_imgs = G_collapse(noise)
        d_fake = D_collapse(fake_imgs.detach())
        d_fake_loss = criterion_fc(d_fake.squeeze(), torch.zeros(bs, device=DEVICE))
        (d_real_loss + d_fake_loss).backward()
        d_opt_collapse.step()

        # Train G
        g_opt_collapse.zero_grad()
        noise = torch.randn(bs, 2, device=DEVICE)
        fake_imgs = G_collapse(noise)
        d_fake = D_collapse(fake_imgs)
        g_loss = criterion_fc(d_fake.squeeze(), torch.ones(bs, device=DEVICE))
        g_loss.backward()
        g_opt_collapse.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch} done")

In [ ]:
# Show collapsed vs healthy generated images
G_collapse.eval()
with torch.no_grad():
    noise_collapsed = torch.randn(25, 2, device=DEVICE)
    collapsed_imgs = G_collapse(noise_collapsed).cpu()
    collapsed_imgs = (collapsed_imgs + 1) * 0.5

# Compute diversity scores
diversity_collapsed = hp.diversity_score(collapsed_imgs)
diversity_healthy = hp.diversity_score(healthy_fc_generated)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

# Collapsed
grid_collapsed = make_grid(collapsed_imgs, nrow=5).permute(1, 2, 0)
ax1.imshow(grid_collapsed.clip(0, 1), cmap='gray')
ax1.set_title(f"Mode Collapse (z_dim=2)\nDiversity: {diversity_collapsed:.2f}")
ax1.axis('off')

# Healthy
grid_healthy = make_grid(healthy_fc_generated, nrow=5).permute(1, 2, 0)
ax2.imshow(grid_healthy.clip(0, 1), cmap='gray')
ax2.set_title(f"Healthy GAN (z_dim=100)\nDiversity: {diversity_healthy:.2f}")
ax2.axis('off')

plt.suptitle("Mode Collapse vs Healthy GAN — MNIST")
plt.tight_layout()
plt.show()

print(f"Diversity score (collapsed): {diversity_collapsed:.2f}")
print(f"Diversity score (healthy):   {diversity_healthy:.2f}")

## Summary

Classical GANs suffer from two major problems:
1. **Training instability**: Small hyperparameter changes can cause divergence. The minimax game is inherently unstable.
2. **Mode collapse**: The generator may learn to produce only a few types of outputs, ignoring the diversity of the real data.

In **LESSON 3B**, we'll see how the **Wasserstein GAN with Gradient Penalty (WGAN-GP)** addresses these problems by replacing the BCE loss with the Wasserstein distance, providing smoother gradients and more stable training.